In [2]:
import numpy as np
import re
from urllib.parse import urlparse
from typing import List, Dict, Tuple

def extract_source_features(text: str) -> Dict[str, float]:
    """Extract features related to source credibility"""
    features = {
        'has_url': 0.0,
        'has_credible_domain': 0.0,
        'has_quotes': 0.0,
        'has_numbers': 0.0,
        'has_date': 0.0
    }
    
    # Check for URLs
    urls = re.findall(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', text)
    if urls:
        features['has_url'] = 1.0
        # List of credible Thai news domains
        credible_domains = {'thairath.co.th', 'bangkokpost.com', 'nationthailand.com', 
                          'matichon.co.th', 'dailynews.co.th', 'prachachat.net'}
        for url in urls:
            domain = urlparse(url).netloc
            if domain in credible_domains:
                features['has_credible_domain'] = 1.0
                break
    
    # Check for quotes (both Thai and English quotes)
    if re.search(r'["\'“”‘’′″]', text):
        features['has_quotes'] = 1.0
    
    # Check for numbers (potential statistics/facts)
    if re.search(r'\d+', text):
        features['has_numbers'] = 1.0
    
    # Check for dates
    date_patterns = [
        r'\d{1,2}/\d{1,2}/\d{2,4}',  # dd/mm/yyyy
        r'\d{1,2}\s(?:ม.ค.|ก.พ.|มี.ค.|เม.ย.|พ.ค.|มิ.ย.|ก.ค.|ส.ค.|ก.ย.|ต.ค.|พ.ย.|ธ.ค.)\s?\d{2,4}'  # Thai months
    ]
    if any(re.search(pattern, text) for pattern in date_patterns):
        features['has_date'] = 1.0
    
    return features

def extract_language_features(tokens: List[str]) -> Dict[str, float]:
    """Extract features related to language structure"""
    features = {
        'avg_word_length': 0.0,
        'vocab_richness': 0.0,
        'has_formal_words': 0.0
    }
    
    if not tokens:
        return features
        
    # Average word length
    features['avg_word_length'] = np.mean([len(t) for t in tokens])
    
    # Vocabulary richness (unique words / total words)
    features['vocab_richness'] = len(set(tokens)) / len(tokens)
    
    # Check for formal words/particles (Thai formal particles and words)
    formal_words = {'ครับ', 'ค่ะ', 'จะ', 'ท่าน', 'กระผม', 'ดิฉัน'}
    if any(word in formal_words for word in tokens):
        features['has_formal_words'] = 1.0
    
    return features

def extract_factual_features(text: str, tokens: List[str]) -> Dict[str, float]:
    """Extract features related to factual accuracy"""
    features = {
        'has_person_title': 0.0,
        'has_location': 0.0,
        'has_organization': 0.0,
        'has_numbers_with_units': 0.0
    }
    
    # Thai person titles
    person_titles = {'นาย', 'นาง', 'นางสาว', 'ดร.', 'อาจารย์', 'รศ.', 'ศ.', 'พล.อ.', 'พ.ต.อ.'}
    if any(title in text for title in person_titles):
        features['has_person_title'] = 1.0
    
    # Common Thai location words
    location_indicators = {'จังหวัด', 'อำเภอ', 'ตำบล', 'เขต', 'ถนน', 'ซอย'}
    if any(loc in text for loc in location_indicators):
        features['has_location'] = 1.0
    
    # Common Thai organization words
    org_indicators = {'บริษัท', 'องค์กร', 'สำนักงาน', 'กระทรวง', 'กรม', 'มหาวิทยาลัย'}
    if any(org in text for org in org_indicators):
        features['has_organization'] = 1.0
    
    # Numbers with units
    unit_patterns = [
        r'\d+\s*(?:บาท|เปอร์เซ็นต์|%|กิโลเมตร|กม\.|เมตร|ม\.|กิโลกรัม|กก\.)',
    ]
    if any(re.search(pattern, text) for pattern in unit_patterns):
        features['has_numbers_with_units'] = 1.0
    
    return features

def extract_cross_source_features(text: str) -> Dict[str, float]:
    """Extract features related to cross-source consistency"""
    features = {
        'has_reference': 0.0,
        'multiple_sources': 0.0,
        'has_comparison': 0.0
    }
    
    # Reference indicators
    reference_words = {'อ้างอิง', 'ตาม', 'จาก', 'โดย', 'ระบุว่า', 'เปิดเผยว่า'}
    if any(ref in text for ref in reference_words):
        features['has_reference'] = 1.0
    
    # Multiple source indicators
    source_indicators = {'แหล่งข่าว', 'ผู้สื่อข่าว', 'นักข่าว', 'สื่อ', 'รายงาน'}
    source_count = sum(1 for s in source_indicators if s in text)
    if source_count > 1:
        features['multiple_sources'] = 1.0
    
    # Comparison indicators
    comparison_words = {'เช่นเดียวกับ', 'สอดคล้องกับ', 'เหมือนกับ', 'ตรงกับ'}
    if any(comp in text for comp in comparison_words):
        features['has_comparison'] = 1.0
    
    return features

# 🔄 ทดสอบ Tokenizer ใหม่

ทดสอบ tokenizer แบบใหม่ที่ปรับปรุงให้:
1. ใช้ attacut (ถ้าติดตั้ง) หรือ fallback ไป newmm
2. กรองตัวเลขและคำสั้นออก
3. ทำความสะอาดข้อความดีขึ้น

In [3]:
import joblib

X_under = joblib.load('../InitData/result/X_under.pkl')


In [4]:
import re
import pandas as pd
from tqdm import tqdm
from pythainlp import word_tokenize
from pythainlp.corpus.common import thai_stopwords
from sklearn.feature_extraction import text

# Prepare stopwords set (reuse existing logic)
thai_stopwords_set = set(thai_stopwords())
english_stopwords_set = set(text.ENGLISH_STOP_WORDS)
thai_stopwords_set.update([
    'ของ', 'ใน', 'ที่', 'จาก', 'ไป', 'ด้วย', 'กับ', 'และ', 'หรือ', 'แต่', 'เพราะ',
    'ถ้า', 'หาก', 'เขา', 'คุณ', 'ฉัน', 'เรา', 'ทุกคน', 'นี้', 'นั้น', 'คือ', 'การ',
    'ได้', 'มี', 'เป็น', 'ซึ่ง', 'ว่า', 'ก็', 'โดย', 'เช่น', 'เพื่อ'
])
all_stopwords = thai_stopwords_set.union(english_stopwords_set)

# Improved clean_text: keep only Thai/Eng/digits and normalize whitespace
def clean_text_improved(text_input: str) -> str:
    s = str(text_input)
    # unify unicode spaces and remove control chars
    s = s.replace('\u00A0', ' ').replace('\u200b', ' ').strip()
    s = s.lower().strip()
    # remove URLs / mentions / hashtags
    s = re.sub(r'http\S+|www\S+|https\S+', ' ', s)
    s = re.sub(r'@\w+|#\w+', ' ', s)
    # keep Thai chars (ก-๙), latin, digits and spaces; replace others with space
    s = re.sub(r'[^ก-๙a-zA-Z0-9\s]', ' ', s)
    # collapse multiple spaces
    s = re.sub(r'\s+', ' ', s).strip()
    return s

# Improved hybrid tokenizer: try attacut (if installed) for Thai, else fallback to pythainlp newmm
def hybrid_tokenizer_improved(text_input: str, tokenizer_preference: str = 'auto'):
    s = clean_text_improved(text_input)
    # split into contiguous Thai / English / digits segments
    segments = re.findall(r'[a-zA-Z]+|[ก-๙]+|\d+', s)

    # try to import attacut (optional, gives often better Thai segmentation)
    attacut_tokenize = None
    if tokenizer_preference in ('auto', 'attacut'):
        try:
            import attacut
            attacut_tokenize = attacut.tokenize
        except Exception:
            attacut_tokenize = None

    tokens = []
    for seg in segments:
        if re.match(r'[ก-๙]+', seg):
            # Thai segmentation
            if tokenizer_preference == 'attacut' and attacut_tokenize is None:
                toks = word_tokenize(seg, engine='newmm')
            else:
                if attacut_tokenize is not None:
                    toks = attacut_tokenize(seg)
                else:
                    toks = word_tokenize(seg, engine='newmm')
            tokens.extend(toks)
        else:
            # english or numeric tokens
            tokens.append(seg.lower())

    # filter tokens: remove stopwords, single-char, and pure digits
    filtered = []
    for w in tokens:
        if not w:
            continue
        if w in all_stopwords:
            continue
        if len(w) <= 1:
            continue
        if re.fullmatch(r'\d+', w):
            continue
        filtered.append(w)

    return filtered


## 🔄 อัปเดต Pipeline

หลังจากทดสอบ tokenizer ใหม่แล้ว ถ้าต้องการใช้ tokenizer ใหม่กับข้อมูลทั้งหมด ให้รันเซลล์ด้านล่างเพื่ออัปเดต pipeline:

1. อัปเดตคอลัมน์ `Tokens` และ `TokenStr` ด้วย tokenizer ใหม่
2. สร้าง TF-IDF matrix ใหม่
3. บันทึกไฟล์ผลลัพธ์

⚠️ หมายเหตุ: 
- แนะนำให้ติดตั้ง attacut ก่อน (`pip install attacut`) เพื่อให้ได้ผลลัพธ์ที่ดีที่สุด
- การรันใหม่จะใช้เวลาสักครู่เนื่องจากต้อง tokenize ข้อมูลทั้งหมดอีกครั้ง

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import numpy as np
import pandas as pd
from scipy.sparse import hstack

# อัปเดต pipeline ด้วย tokenizer ใหม่และ features เพิ่มเติม
print("🔄 เริ่มอัปเดต pipeline...")

# 1. อัปเดต Tokens column
print("\n1️⃣ Tokenizing ข้อมูลใหม่...")
X_under["Text"] = X_under["Title"].fillna("") + " " + X_under["Body"].fillna("")

# ใช้ list comprehension with tqdm แทน progress_apply
print("กำลัง tokenize ข้อความ...")
tokens = [hybrid_tokenizer_improved(text, tokenizer_preference='attacut') 
         for text in tqdm(X_under["Text"], desc="Tokenizing")]
X_under["Tokens"] = tokens
X_under["TokenStr"] = X_under["Tokens"].apply(lambda x: " ".join(x))

# 2. สร้าง TF-IDF matrix
print("\n2️⃣ สร้าง TF-IDF matrix...")
vectorizer = TfidfVectorizer(
    analyzer="word",
    max_df=0.9,
    min_df=1,
    stop_words=list(all_stopwords),
    ngram_range=(1, 2),
    max_features=10000
)

X_tfidf = vectorizer.fit_transform(X_under["TokenStr"])

# 3. สร้าง features เพิ่มเติม
print("\n3️⃣ สร้าง features เพิ่มเติม...")

# Extract all features
additional_features = []
for idx in tqdm(range(len(X_under)), desc="Extracting features"):
    text = X_under.iloc[idx]["Text"]
    tokens_list = X_under.iloc[idx]["Tokens"]
    
    # Get all features
    source_feats = extract_source_features(text)
    lang_feats = extract_language_features(tokens_list)
    fact_feats = extract_factual_features(text, tokens_list)
    cross_feats = extract_cross_source_features(text)
    
    # Combine all features
    combined_feats = {**source_feats, **lang_feats, **fact_feats, **cross_feats}
    additional_features.append(list(combined_feats.values()))

# Convert to numpy array and scale
X_additional = np.array(additional_features)
scaler = StandardScaler()
X_additional_scaled = scaler.fit_transform(X_additional)

# Combine TF-IDF with additional features
X_combined = hstack([X_tfidf, X_additional_scaled])

# แสดงผล
print("\n✅ สร้าง features สำเร็จ!")
print("Shape TF-IDF:", X_tfidf.shape)
print("Shape additional features:", X_additional_scaled.shape)
print("Shape combined:", X_combined.shape)

# แสดงตัวอย่าง features
feature_names = {
    'Source Credibility': ['has_url', 'has_credible_domain', 'has_quotes', 'has_numbers', 'has_date'],
    'Language Structure': ['avg_word_length', 'vocab_richness', 'has_formal_words'],
    'Factual Accuracy': ['has_person_title', 'has_location', 'has_organization', 'has_numbers_with_units'],
    'Cross-source': ['has_reference', 'multiple_sources', 'has_comparison']
}

print("\n🔸 ตัวอย่างค่าเฉลี่ย features แต่ละกลุ่ม:")
for group, features in feature_names.items():
    start_idx = 0
    for prev_group, prev_features in list(feature_names.items())[:list(feature_names.keys()).index(group)]:
        start_idx += len(prev_features)
    end_idx = start_idx + len(features)
    
    print(f"\n{group}:")
    means = np.mean(X_additional, axis=0)[start_idx:end_idx]
    for feat, mean in zip(features, means):
        print(f"- {feat}: {mean:.3f}")

🔄 เริ่มอัปเดต pipeline...

1️⃣ Tokenizing ข้อมูลใหม่...
กำลัง tokenize ข้อความ...


Tokenizing: 100%|██████████| 4245/4245 [1:25:56<00:00,  1.21s/it]



2️⃣ สร้าง TF-IDF matrix...


C:\Users\SUPHASET\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\feature_extraction\text.py:406: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['กคน', 'กคร', 'กครา', 'กคราว', 'กจะ', 'กช', 'กต', 'กท', 'กทาง', 'กน', 'กระท', 'กระน', 'กระไร', 'กล', 'กว', 'กส', 'กหน', 'กอ', 'กอย', 'กำล', 'กเม', 'กแห', 'กๆ', 'ขณะท', 'ขณะน', 'ขณะหน', 'ขณะเด', 'คงอย', 'คร', 'ครบคร', 'ครบถ', 'คราท', 'คราน', 'คราวก', 'คราวท', 'คราวน', 'คราวหน', 'คราวหล', 'คราวโน', 'คราหน', 'คล', 'งก', 'งกระน', 'งกล', 'งกว', 'งข', 'งคง', 'งคน', 'งครา', 'งคราว', 'งง', 'งจ', 'งจน', 'งจะ', 'งจาก', 'งต', 'งท', 'งน', 'งบ', 'งปวง', 'งมวล', 'งละ', 'งว', 'งส', 'งหน', 'งหมด', 'งหมาย', 'งหล', 'งหลาย', 'งอย', 'งเก', 'งเคย', 'งเน', 'งเป', 'งเม', 'งแก', 'งแต', 'งแม', 'งแล', 'งโง', 'งโน', 'งใด', 'งใหญ', 'งไง', 'งได', 'งไหน', 'งๆ', 'งๆจ', 'จก', 'จจ', 'จนกระท', 'จนกว', 'จนขณะน', 'จนถ', 'จนท',


3️⃣ สร้าง features เพิ่มเติม...


Extracting features: 100%|██████████| 4245/4245 [00:01<00:00, 2357.33it/s]


✅ สร้าง features สำเร็จ!
Shape TF-IDF: (4245, 10000)
Shape additional features: (4245, 15)
Shape combined: (4245, 10015)

🔸 ตัวอย่างค่าเฉลี่ย features แต่ละกลุ่ม:

Source Credibility:
- has_url: 0.298
- has_credible_domain: 0.000
- has_quotes: 0.203
- has_numbers: 0.987
- has_date: 0.153

Language Structure:
- avg_word_length: 5.745
- vocab_richness: 0.522
- has_formal_words: 0.031

Factual Accuracy:
- has_person_title: 0.280
- has_location: 0.234
- has_organization: 0.934
- has_numbers_with_units: 0.330

Cross-source:
- has_reference: 0.995
- multiple_sources: 0.038
- has_comparison: 0.067


In [8]:
# บันทึกผลลัพธ์
print("3️⃣ บันทึกผลลัพธ์...")

# บันทึก TF-IDF vectorizer
joblib.dump(vectorizer, 'result/vectorizer-new.pkl')

# บันทึก X_tfidf (sparse matrix)
joblib.dump(X_tfidf, 'result/X_tfidf-new.pkl')
joblib.dump(X_additional, 'result/X_additional-new.pkl')

print("✅ บันทึกข้อมูลเรียบร้อยแล้ว!")
print("\nℹ️ หมายเหตุ: ถ้าต้องการใช้ผลลัพธ์ใหม่ ให้รัน notebook อื่น ๆ ใหม่ด้วย (NeuralNetwork.ipynb)")

3️⃣ บันทึกผลลัพธ์...
✅ บันทึกข้อมูลเรียบร้อยแล้ว!

ℹ️ หมายเหตุ: ถ้าต้องการใช้ผลลัพธ์ใหม่ ให้รัน notebook อื่น ๆ ใหม่ด้วย (NeuralNetwork.ipynb)
